# 15 · `gl_engine/rating/submission.py`

## What this file is for

A submission arrives as JSON. ISO's rules read a tree. This is the mapping — 122 lines, and most of the subtlety is in one idea.

ISO's rules address repeated things with `ForEach`, which needs each list to be a **container of numbered children**, not a bare list. So a JSON array of locations becomes an `XTable` holding `X` elements. Get that wrong and every multi-location risk silently prices as its first location.

**Depends on:** [`09-interp-tree`](09-interp-tree.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.rating import submission

for name, obj in vars(submission).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != submission.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Load a submission and see what came back.

In [ ]:
from gl_engine.rating import submission

tree, juris, asof = submission.load("../Engine_Payloads/GA/submission.json")

print("jurisdiction:", juris)
print("as-of       :", asof)
print("root tag    :", tree.tag)

Three things come out, not one: the tree, **and the jurisdiction and date read out of the submission itself**. That is what makes the rating self-describing — nothing external has to say which state this is.

## The interesting case

### Lists become containers

In [ ]:
from gl_engine.interp import tree as T

text = T.dump(tree)
lines = [l for l in text.splitlines() if "XTable" in l or "<X" in l or "X>" in l]
print(f"{len(lines)} container/element lines in this submission\n")
for l in lines[:12]:
    print(l)

### The same shape, from a raw dict

`from_raas` is the one that takes a payload you already have in memory — which is what the comparison scripts use when they build variants.

In [ ]:
import json
from pathlib import Path

payload = json.loads(Path("../Engine_Payloads/GA/submission.json").read_text())
t2, j2, a2 = submission.from_raas(payload)

print("same jurisdiction:", j2 == juris)
print("same as-of       :", a2 == asof)
print("same shape       :", T.dump(t2)[:60] == T.dump(tree)[:60])

### What the rules will actually read

In [ ]:
for path in ("GeneralLiability/StateCode",
             "GeneralLiability/EffDate",
             "GeneralLiability/GeneralLiabilityLocation"):
    hits = T.select(path, tree)
    val = T.read(path, tree)
    print(f"{path:<46} {len(hits)} match(es)  {val!r}")

## What it refuses

A payload that doesn't say which jurisdiction or date it is.

In [ ]:
try:
    submission.from_raas({"nothing": "useful"})
    print("no error")
except Exception as e:
    print(f"{type(e).__name__}: {str(e)[:140]}")

## Try it yourself

1. Add a second location to the payload and reload. How many `X` elements appear?
2. Compare `Engine_Payloads/GA` with `Engine_Payloads/NY`. What differs beyond the state code?
3. Find where `XTable` is written in the source. Why a container rather than a flat list?

In [ ]:
# your turn